# Deploy Parakeet ASR with NVIDIA Riva on GKE — Realtime Streaming

This notebook deploys the [NVIDIA Parakeet 1.1B CTC](https://build.nvidia.com/nvidia/parakeet-1-1b-ctc-en-us) model on **Google Kubernetes Engine (GKE)** using **NVIDIA Riva NIM** for realtime streaming speech recognition.

| | |
|---|---|
| **Task** | Realtime streaming ASR with End-of-Utterance detection |
| **Model** | Parakeet 1.1B CTC English |
| **Parameters** | 1.1 B |
| **Language** | English |
| **Input** | 16 kHz mono audio (WAV, OPUS, FLAC) |
| **Output** | Streaming partial + final transcriptions |
| **Inference** | NVIDIA Riva NIM (TensorRT-optimized) |
| **EOU** | Two-Pass EOU + Silero VAD endpointing |

### Why GKE + Riva instead of Vertex AI?

| Feature | Vertex AI / Cloud Run | GKE + Riva NIM |
|---------|----------------------|----------------|
| Streaming | No (HTTP only) | **gRPC + WebSocket** |
| Realtime partial results | No | **Yes** |
| EOU detection | Manual (`<EOU>` tokens) | **Built-in (Two-Pass EOU + VAD)** |
| Model optimization | PyTorch (NeMo) | **TensorRT (2-5x faster)** |
| Latency | ~1-2s per request | **~200ms streaming** |

**What this notebook does:**
1. Creates a GKE cluster with an NVIDIA T4 GPU node
2. Deploys the Riva NIM Parakeet ASR container via Helm
3. Tests offline transcription via HTTP
4. Tests realtime streaming via gRPC and WebSocket
5. Cleans up all resources

## Prerequisites

Before running this notebook, ensure you have:

1. A **Google Cloud project** with billing enabled
2. The following **APIs enabled**:
   - [Kubernetes Engine API](https://console.cloud.google.com/apis/api/container.googleapis.com)
   - [Compute Engine API](https://console.cloud.google.com/apis/api/compute.googleapis.com)
3. **GPU quota**: At least 1x NVIDIA T4 in your target zone (check [quotas](https://console.cloud.google.com/iam-admin/quotas))
4. An **NVIDIA NGC API key** (free — sign up at [ngc.nvidia.com](https://ngc.nvidia.com))
5. Your account has **IAM roles**: `Kubernetes Engine Admin`, `Compute Admin`

In [ ]:
!pip install -q nvidia-riva-client gTTS pydub websocket-client

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated via Colab OAuth.")
else:
    print(
        "Not running in Colab.\n"
        "Make sure Application Default Credentials are set:\n"
        "  gcloud auth application-default login"
    )

In [ ]:
# ============================================================
#  CONFIGURATION — update these for your project
# ============================================================
PROJECT_ID  = "your-project-id"    # @param {type:"string"}
ZONE        = "us-central1-a"      # @param {type:"string"}
NGC_API_KEY = "your-ngc-api-key"   # @param {type:"string"}

# Cluster settings
CLUSTER_NAME = "riva-asr-cluster"
MACHINE_TYPE = "n1-standard-8"
GPU_TYPE     = "nvidia-tesla-t4"
NUM_NODES    = 1

# Riva NIM settings
RIVA_MODEL       = "parakeet-1-1b-ctc-en-us"
RIVA_IMAGE        = f"nvcr.io/nim/nvidia/{RIVA_MODEL}"
RIVA_TAG          = "latest"
NIM_TAGS_SELECTOR = f"name={RIVA_MODEL},mode=str,vad=default"

# Point gcloud at the right project
!gcloud config set project {PROJECT_ID} --quiet
!gcloud config set compute/zone {ZONE} --quiet

print(f"Project:  {PROJECT_ID}")
print(f"Zone:     {ZONE}")
print(f"Cluster:  {CLUSTER_NAME}")
print(f"Model:    {RIVA_MODEL}")
print(f"GPU:      {GPU_TYPE}")

## Step 1 — Create a GKE Cluster with GPU

We create a GKE cluster with a single GPU node pool running an NVIDIA T4.

```
Architecture:
┌──────────────────────────────────────────────────┐
│  GKE Cluster                                     │
│  └── GPU Node Pool (n1-standard-8 + T4)          │
│      └── Pod: riva-nim                           │
│          └── Container: parakeet-1-1b-ctc-en-us  │
│              ├── HTTP  :9000  → health + offline  │
│              ├── gRPC  :50051 → streaming ASR     │
│              └── WS    :9000  → realtime API      │
└──────────────────────────────────────────────────┘
```

**Estimated cost:** ~$0.83/hr (cluster mgmt + n1-standard-8 + T4 GPU)

In [ ]:
# Create the GKE cluster with a GPU node pool.
# This typically takes 3-5 minutes.
print(f"Creating GKE cluster '{CLUSTER_NAME}' with {GPU_TYPE} GPU ...")
print("This will take 3-5 minutes.\n")

!gcloud container clusters create {CLUSTER_NAME} \
    --zone {ZONE} \
    --machine-type {MACHINE_TYPE} \
    --accelerator type={GPU_TYPE},count=1 \
    --num-nodes {NUM_NODES} \
    --scopes cloud-platform \
    --quiet

# Get credentials for kubectl
!gcloud container clusters get-credentials {CLUSTER_NAME} --zone {ZONE} --quiet

print("\nCluster created. Verifying nodes ...")
!kubectl get nodes

In [ ]:
# Install NVIDIA GPU device drivers on the cluster nodes.
# GKE provides a daemonset for this.
print("Installing NVIDIA GPU drivers ...")

!kubectl apply -f https://raw.githubusercontent.com/GoogleCloudPlatform/container-engine-accelerators/master/nvidia-driver-installer/cos/daemonset-preloaded-latest.yaml

# Wait for the driver installer to complete
import time
print("Waiting for GPU drivers to initialize (60s) ...")
time.sleep(60)

# Verify GPU is visible
!kubectl get nodes -o=custom-columns='NAME:.metadata.name,GPU:.status.allocatable.nvidia\.com/gpu'

In [ ]:
# Create Kubernetes secrets for NGC authentication.
# These allow the cluster to pull the Riva NIM container and download models.

# Image pull secret — for pulling from nvcr.io
!kubectl create secret docker-registry ngc-secret \
    --docker-server=nvcr.io \
    --docker-username='$oauthtoken' \
    --docker-password={NGC_API_KEY} \
    --dry-run=client -o yaml | kubectl apply -f -

# API key secret — for downloading model weights
!kubectl create secret generic ngc-api \
    --from-literal=NGC_API_KEY={NGC_API_KEY} \
    --dry-run=client -o yaml | kubectl apply -f -

print("NGC secrets created.")

## Step 2 — Deploy Riva NIM via Helm

We deploy the Riva NIM container using NVIDIA's official Helm chart. The container will:

1. Pull the Parakeet 1.1B CTC model from NGC
2. Optimize it with TensorRT for the T4 GPU
3. Start the gRPC (port 50051) and HTTP (port 9000) endpoints

**First deployment takes 15-30 minutes** (model download + TensorRT optimization).
Subsequent restarts are much faster if a PersistentVolume is used for caching.

In [ ]:
# Install Helm if not present
!which helm || (curl https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash)

# Add the NVIDIA NIM Helm repository
!helm repo add nim https://helm.ngc.nvidia.com/nim/nvidia \
    --username='$oauthtoken' --password={NGC_API_KEY} 2>/dev/null; \
  helm repo update nim

# List available Riva charts
!helm search repo nim/riva-nim

In [ ]:
%%writefile /tmp/riva-values.yaml
image:
  repository: nvcr.io/nim/nvidia/parakeet-1-1b-ctc-en-us
  tag: latest
  pullPolicy: IfNotPresent

imagePullSecrets:
  - name: ngc-secret

nim:
  ngcAPISecret: ngc-api

env:
  - name: NIM_TAGS_SELECTOR
    value: "name=parakeet-1-1b-ctc-en-us,mode=str,vad=default"

resources:
  limits:
    nvidia.com/gpu: 1
  requests:
    nvidia.com/gpu: 1

service:
  type: LoadBalancer
  ports:
    - name: http
      port: 9000
      targetPort: 9000
      protocol: TCP
    - name: grpc
      port: 50051
      targetPort: 50051
      protocol: TCP

persistence:
  enabled: true
  size: 50Gi

startupProbe:
  httpGet:
    path: /v1/health/ready
    port: 9000
  initialDelaySeconds: 60
  periodSeconds: 30
  failureThreshold: 40
  timeoutSeconds: 10

livenessProbe:
  httpGet:
    path: /v1/health/ready
    port: 9000
  periodSeconds: 30
  failureThreshold: 3
  timeoutSeconds: 10

In [ ]:
# Deploy Riva NIM
print("Deploying Riva NIM with Parakeet 1.1B CTC ...")
print("First deployment takes 15-30 min (model download + TensorRT optimization).\n")

!helm install riva-asr nim/riva-nim -f /tmp/riva-values.yaml

print("\nHelm release created. Monitoring pod status ...")

In [ ]:
import subprocess, time, json

def get_pod_status():
    result = subprocess.run(
        ["kubectl", "get", "pods", "-l", "app.kubernetes.io/instance=riva-asr",
         "-o", "json"],
        capture_output=True, text=True
    )
    pods = json.loads(result.stdout).get("items", [])
    if not pods:
        return "No pods found", False
    pod = pods[0]
    phase = pod["status"]["phase"]
    ready = any(
        c.get("ready", False)
        for c in pod["status"].get("containerStatuses", [])
    )
    return phase, ready

def get_external_ip():
    result = subprocess.run(
        ["kubectl", "get", "svc", "riva-asr-riva-nim",
         "-o", "jsonpath={.status.loadBalancer.ingress[0].ip}"],
        capture_output=True, text=True
    )
    return result.stdout.strip()

# Wait for the pod to become ready (up to 40 min)
print("Waiting for Riva NIM pod to become ready ...")
print("(Model download + TensorRT optimization in progress)\n")

for i in range(80):
    phase, ready = get_pod_status()
    ip = get_external_ip()
    elapsed = i * 30
    print(f"  [{elapsed//60}m {elapsed%60:02d}s] Phase: {phase}, Ready: {ready}, IP: {ip or 'pending'}")
    if ready and ip:
        break
    time.sleep(30)
else:
    print("\nTimeout — check pod logs with: kubectl logs -l app.kubernetes.io/instance=riva-asr")

EXTERNAL_IP = get_external_ip()
print(f"\nRiva NIM is ready!")
print(f"  HTTP:      http://{EXTERNAL_IP}:9000")
print(f"  gRPC:      {EXTERNAL_IP}:50051")
print(f"  WebSocket: ws://{EXTERNAL_IP}:9000/v1/realtime?intent=transcription")

In [ ]:
import urllib.request, json

# Verify the service is healthy
url = f"http://{EXTERNAL_IP}:9000/v1/health/ready"
req = urllib.request.Request(url)
with urllib.request.urlopen(req, timeout=10) as resp:
    result = json.loads(resp.read())
    print(f"Health check: {result}")
    assert result.get("ready") is True, "Service is not ready!"
    print("Riva NIM is healthy and ready for inference.")

## Step 3 — Test the Deployed Model

We test three access methods:
1. **HTTP offline** — send a complete WAV file, get full transcription
2. **gRPC streaming** — stream audio chunks, get interim + final results
3. **WebSocket realtime** — browser-friendly streaming via WebSocket

In [ ]:
import base64
from gtts import gTTS
from pydub import AudioSegment

def text_to_wav(text: str, path: str = "/tmp/test_riva.wav") -> str:
    """Convert text to a 16 kHz mono WAV file via gTTS."""
    tts = gTTS(text, lang="en")
    mp3_path = path.replace(".wav", ".mp3")
    tts.save(mp3_path)
    audio = (
        AudioSegment.from_mp3(mp3_path)
        .set_frame_rate(16000)
        .set_channels(1)
        .set_sample_width(2)  # 16-bit
    )
    audio.export(path, format="wav")
    return path

# Generate test audio
test_text = "Hello, what is the weather like today?"
wav_path = text_to_wav(test_text)
print(f'Test audio generated: "{test_text}"')
print(f"File: {wav_path}")

In [ ]:
import urllib.request, json

# Test 1: HTTP offline transcription
print("=" * 60)
print("TEST 1: HTTP Offline Transcription")
print("=" * 60)

# Build multipart form data
import io, uuid

boundary = uuid.uuid4().hex
body = io.BytesIO()

# Add language field
body.write(f"--{boundary}\r\n".encode())
body.write(b'Content-Disposition: form-data; name="language"\r\n\r\n')
body.write(b"en\r\n")

# Add audio file
body.write(f"--{boundary}\r\n".encode())
body.write(b'Content-Disposition: form-data; name="file"; filename="test.wav"\r\n')
body.write(b"Content-Type: audio/wav\r\n\r\n")
with open(wav_path, "rb") as f:
    body.write(f.read())
body.write(b"\r\n")
body.write(f"--{boundary}--\r\n".encode())

data = body.getvalue()

req = urllib.request.Request(
    f"http://{EXTERNAL_IP}:9000/v1/audio/transcriptions",
    data=data,
    headers={"Content-Type": f"multipart/form-data; boundary={boundary}"},
)

with urllib.request.urlopen(req, timeout=60) as resp:
    result = json.loads(resp.read())

print(f"  Input:         \"{test_text}\"")
print(f"  Transcription: \"{result.get('text', '')}\"")
print(f"  Response:      {json.dumps(result, indent=2)}")

In [ ]:
# Test 2: gRPC Streaming Transcription
print("=" * 60)
print("TEST 2: gRPC Streaming Transcription")
print("=" * 60)

import riva.client

# Connect to Riva gRPC endpoint
auth = riva.client.Auth(uri=f"{EXTERNAL_IP}:50051", use_ssl=False)
asr_service = riva.client.ASRService(auth)

# Configure streaming recognition
config = riva.client.StreamingRecognitionConfig(
    config=riva.client.RecognitionConfig(
        language_code="en-US",
        max_alternatives=1,
        enable_automatic_punctuation=True,
        verbatim_transcripts=False,
    ),
    interim_results=True,
)

# Read audio file and stream it in chunks
CHUNK_SIZE = 8000  # 250ms of 16kHz 16-bit mono audio

with open(wav_path, "rb") as audio_file:
    # Skip WAV header (44 bytes)
    audio_file.read(44)
    audio_chunks = []
    while True:
        chunk = audio_file.read(CHUNK_SIZE)
        if not chunk:
            break
        audio_chunks.append(chunk)

print(f"  Streaming {len(audio_chunks)} audio chunks ...\n")

# Stream audio and collect responses
responses = asr_service.streaming_response_generator(
    audio_chunks=audio_chunks,
    streaming_config=config,
)

final_transcript = ""
for response in responses:
    for result in response.results:
        transcript = result.alternatives[0].transcript
        if result.is_final:
            final_transcript += transcript
            print(f"  [FINAL]   {transcript}")
        else:
            print(f"  [INTERIM] {transcript}")

print(f"\n  Full transcription: \"{final_transcript}\"")

In [ ]:
# Test 3: WebSocket Realtime Transcription
print("=" * 60)
print("TEST 3: WebSocket Realtime Transcription")
print("=" * 60)

import websocket
import json
import base64
import threading

ws_url = f"ws://{EXTERNAL_IP}:9000/v1/realtime?intent=transcription"
print(f"  Connecting to {ws_url} ...\n")

results = []
done_event = threading.Event()

def on_message(ws, message):
    event = json.loads(message)
    event_type = event.get("type", "")
    if event_type == "conversation.item.input_audio_transcription.delta":
        text = event.get("delta", "")
        print(f"  [PARTIAL] {text}")
        results.append(("partial", text))
    elif event_type == "conversation.item.input_audio_transcription.completed":
        text = event.get("transcript", "")
        print(f"  [FINAL]   {text}")
        results.append(("final", text))
        done_event.set()
    elif event_type == "error":
        print(f"  [ERROR]   {event}")
        done_event.set()

def on_open(ws):
    # Configure session
    config_msg = {
        "type": "transcription_session.update",
        "input_audio_format": "pcm16",
        "language": "en-US",
    }
    ws.send(json.dumps(config_msg))

    # Read and send audio
    with open(wav_path, "rb") as f:
        f.read(44)  # skip WAV header
        while True:
            chunk = f.read(8000)
            if not chunk:
                break
            audio_msg = {
                "type": "input_audio_buffer.append",
                "audio": base64.b64encode(chunk).decode("utf-8"),
            }
            ws.send(json.dumps(audio_msg))

    # Signal end of audio
    ws.send(json.dumps({"type": "input_audio_buffer.commit"}))
    ws.send(json.dumps({"type": "input_audio_buffer.done"}))

def on_error(ws, error):
    print(f"  [WS ERROR] {error}")
    done_event.set()

ws = websocket.WebSocketApp(
    ws_url,
    on_open=on_open,
    on_message=on_message,
    on_error=on_error,
)

# Run WebSocket in a background thread
ws_thread = threading.Thread(target=ws.run_forever, kwargs={"ping_interval": 30})
ws_thread.daemon = True
ws_thread.start()

# Wait for transcription to complete (timeout 30s)
done_event.wait(timeout=30)
ws.close()

if results:
    final_texts = [t for kind, t in results if kind == "final"]
    print(f"\n  Final transcription: \"{' '.join(final_texts)}\"")
else:
    print("\n  No results received (timeout or connection issue)")

## Step 4 — Clean Up Resources

**Important:** The GKE cluster with a T4 GPU costs ~$0.83/hr. Run the cell
below to delete everything when you are done.

In [ ]:
# Uninstall Riva Helm release
!helm uninstall riva-asr

# Delete the GKE cluster (this also removes all node pools, disks, etc.)
print(f"Deleting GKE cluster '{CLUSTER_NAME}' ...")
!gcloud container clusters delete {CLUSTER_NAME} --zone {ZONE} --quiet

print("\nAll GKE resources cleaned up — no further charges.")